# Model No.X: Artificial Neural Network (ANN) for Multi-Class Text Classification

## Objective
This section implements an **Artificial Neural Network (ANN)**, operationalised as a **Multilayer Perceptron (MLP)**, to predict the target variable **`y_bucket`** (multi-class classification). The ANN complements earlier baseline and tree-based models by introducing a **non-linear, gradient-based learner** that can model complex decision boundaries in high-dimensional feature spaces.

## Context in the pipeline
The ANN is trained on the final feature representation **`X_final`** that has already been created in the earlier pipeline (e.g., engineered features and/or reduced-dimensional representations such as SVD-based components). Therefore, the ANN does **not** replace the feature engineering stage; it operates as a downstream predictive model.

## Why an ANN is relevant here
An ANN is appropriate when:
1. **Non-linear relationships** are expected between features and the target.
2. Feature vectors are **dense and numeric**, which is typical after embedding and dimensionality reduction.
3. A flexible function approximator is required to potentially exceed the performance of linear baselines.

In addition, the assignment requires that a neural network is implemented and that **training and validation loss curves** are plotted to determine a suitable number of epochs.


In [ ]:
# Import Liberaries

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt



## Step 1 — Prepare the data for neural network training (scaling + label checks)

### What this step does
This step:
- Applies **feature scaling** using `StandardScaler`
- Ensures the feature matrices have the correct **numeric type** (`float32`)
- Ensures the target labels are **integer class indices** (`int64`)

### How it works
- `StandardScaler()` transforms each feature column so that it has approximately:
  - mean = 0
  - standard deviation = 1  
  based on statistics computed from the **training set only**.
- The scaler is then applied to the validation and test sets using the same transformation.

### Why this step is necessary
Neural networks are trained using **gradient descent**. If features have very different scales (e.g., one feature ranges 0–1 while another ranges 0–10,000), then:
- gradients become poorly conditioned,
- learning becomes unstable or very slow,
- the optimiser may over-update some weights and under-update others.

Fitting the scaler only on the training set is critical to avoid **data leakage**, meaning information from validation/test influencing training.

Finally, `CrossEntropyLoss` in PyTorch expects:
- inputs as floating point tensors (`float32`)
- labels as integer class indices (`int64`)

In [ ]:
# Scaling (fit on train only)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Ensure correct dtypes
X_train_s = X_train_s.astype(np.float32)
X_val_s   = X_val_s.astype(np.float32)
X_test_s  = X_test_s.astype(np.float32)

# Ensure labels are integer class indices
y_train_i = y_train.astype(np.int64)
y_val_i   = y_val.astype(np.int64)
y_test_i  = y_test.astype(np.int64)

n_features = X_train_s.shape[1]
n_classes  = len(np.unique(y_train_i))

print("ANN input features:", n_features)
print("ANN classes:", n_classes)

## Step 2 — Create PyTorch datasets and DataLoaders (mini-batch learning)

### What this step does
This step converts the scaled NumPy arrays into PyTorch tensors and creates:
- `TensorDataset` objects for train/validation/test
- `DataLoader` objects to iterate over the data in **mini-batches**

### How it works
- A `TensorDataset` is a simple container that pairs input tensors `X` with label tensors `y`.
- A `DataLoader` provides an efficient iterator that:
  - yields batches of `(X_batch, y_batch)`
  - optionally shuffles the training data each epoch

### Why this step is necessary
Neural networks are almost always trained with **mini-batch gradient descent**, because:
- it is computationally efficient (vectorised operations on batches),
- it stabilises learning compared to single-sample updates,
- it allows the model to scale to larger datasets.

Shuffling is typically enabled for training to reduce ordering effects and improve generalisation.


In [ ]:
# Torch datasets/loaders

def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

## Step 3 — Define the ANN architecture (Multilayer Perceptron)

### What this step does
This step defines an **MLP** consisting of:
- an input layer (size = number of features)
- one or more hidden layers with **ReLU** activations
- **Dropout** layers for regularisation
- an output layer (size = number of classes)

### How it works
- Each linear layer computes:  
  \[
  \mathbf{h} = \mathbf{W}\mathbf{x} + \mathbf{b}
  \]
- ReLU introduces non-linearity:  
  \[
  \text{ReLU}(z) = \max(0, z)
  \]
- Dropout randomly “turns off” a proportion of hidden units during training to prevent the network from over-relying on specific activations.

The final layer outputs **logits** (unnormalised scores) for each class. These logits are used by `CrossEntropyLoss`, which internally applies a softmax transformation during loss computation.

### Why this step is necessary
- A purely linear model can only represent linear decision boundaries.
- Hidden layers with non-linear activations allow the model to learn complex patterns.
- Dropout reduces overfitting by encouraging more robust internal representations, especially when the model has many parameters relative to the dataset size.


In [ ]:
# Model definition (MLP)

class MLP(nn.Module):
    def __init__(self, n_in, n_out, hidden_sizes=(256, 128), dropout=0.3):
        super().__init__()
        layers = []
        prev = n_in
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, n_out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

## Step 4 — Define the learning objective (loss) and optimisation strategy

### What this step does
This step specifies:
- the **loss function**: `CrossEntropyLoss` for multi-class classification
- the **optimiser**: Adam
- optional **L2 regularisation** via `weight_decay`

### How it works
**Cross-Entropy Loss** compares predicted class distributions with true labels. For a single sample with true class \( y \) and predicted probabilities \( p_k \):
\[
\mathcal{L} = -\log(p_y)
\]

Adam is an adaptive gradient method that maintains moving averages of:
- gradients (first moment)
- squared gradients (second moment)  
This typically leads to faster and more stable convergence than plain SGD.

`weight_decay` adds an L2 penalty to large weights, discouraging overly complex solutions.

### Why this step is necessary
- A classification model requires a principled objective; cross-entropy is standard for multi-class targets.
- Adam generally converges reliably with minimal tuning, making it appropriate for this project.
- Weight decay and dropout reduce overfitting and improve generalisation.


In [ ]:
# Train/eval utilities

def run_epoch(model, loader, criterion, optimizer=None, device="cpu"):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    y_true, y_pred = [], []

    with torch.set_grad_enabled(train_mode):
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * xb.size(0)

            preds = torch.argmax(logits, dim=1)
            y_true.append(yb.detach().cpu().numpy())
            y_pred.append(preds.detach().cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return avg_loss, macro_f1

def train_ann(
    hidden_sizes=(256,128),
    dropout=0.3,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=50,
    patience=7,
    device=None
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    train_loader = make_loader(X_train_s, y_train_i, batch_size=batch_size, shuffle=True)
    val_loader   = make_loader(X_val_s, y_val_i, batch_size=batch_size, shuffle=False)

    model = MLP(n_features, n_classes, hidden_sizes=hidden_sizes, dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    train_losses, val_losses = [], []
    train_f1s, val_f1s = [], []

    best_val_loss = float("inf")
    best_state = None
    no_improve = 0

    for epoch in range(1, max_epochs + 1):
        tr_loss, tr_f1 = run_epoch(model, train_loader, criterion, optimizer=optimizer, device=device)
        va_loss, va_f1 = run_epoch(model, val_loader, criterion, optimizer=None, device=device)

        train_losses.append(tr_loss)
        val_losses.append(va_loss)
        train_f1s.append(tr_f1)
        val_f1s.append(va_f1)

        print(f"Epoch {epoch:02d} | Train loss {tr_loss:.4f} F1 {tr_f1:.4f} | Val loss {va_loss:.4f} F1 {va_f1:.4f}")

        # Early stopping on val loss
        if va_loss < best_val_loss - 1e-4:
            best_val_loss = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    history = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "train_f1": train_f1s,
        "val_f1": val_f1s,
        "best_val_loss": best_val_loss
    }
    return model, history

## Step 6 — Early stopping (selecting a good epoch count)

### What this step does
Early stopping halts training when validation loss stops improving for a fixed number of epochs (`patience`). The best-performing model (lowest validation loss) is retained.

### How it works
- Track the best validation loss seen so far.
- If validation loss improves, save the model state.
- If it does not improve for `patience` consecutive epochs, stop training.
- Restore the saved best model weights at the end.

### Why this step is necessary
Neural networks can overfit if trained too long. Early stopping acts as an effective regularisation technique by:
- preventing unnecessary epochs that harm generalisation,
- reducing computation,
- aligning with the requirement to identify an appropriate epoch count using validation behaviour and loss curves.


## Step 7 — Hyperparameter tuning (controlled search)

### What this step does
This step evaluates a small set of hyperparameter configurations, such as:
- hidden layer sizes
- dropout rate
- learning rate
- batch size
- weight decay

The best configuration is selected based on **validation performance**.

### How it works
For each hyperparameter set:
1. Train the ANN with early stopping
2. Record the validation macro-F1 curve
3. Select the best configuration by the highest validation macro-F1 (or lowest validation loss)

### Why this step is necessary
ANN performance is sensitive to hyperparameters. A controlled search:
- improves performance reliability compared to arbitrary choices,
- provides methodological transparency,
- enables defensible model selection based on validation evidence rather than test results (avoiding test-set overfitting).


In [ ]:
# Minimal hyperparameter search

param_grid = [
    {"hidden_sizes": (128,),      "dropout": 0.2, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 64},
    {"hidden_sizes": (256,128),   "dropout": 0.3, "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 64},
    {"hidden_sizes": (256,128),   "dropout": 0.4, "lr": 3e-4, "weight_decay": 1e-3, "batch_size": 64},
]

best = {"val_f1": -1, "model": None, "history": None, "params": None}

for p in param_grid:
    print("\nTesting params:", p)
    model, hist = train_ann(
        hidden_sizes=p["hidden_sizes"],
        dropout=p["dropout"],
        lr=p["lr"],
        weight_decay=p["weight_decay"],
        batch_size=p["batch_size"],
        max_epochs=60,
        patience=8
    )

    # Choose best by max validation macro-F1 (last epoch in history after early stop)
    val_f1 = max(hist["val_f1"])
    if val_f1 > best["val_f1"]:
        best.update({"val_f1": val_f1, "model": model, "history": hist, "params": p})

print("\nBest ANN params:", best["params"])
print("Best ANN Val macro-F1:", best["val_f1"])

## Step 8 — Plot training vs validation loss curves

### What this step does
This step plots:
- training loss per epoch
- validation loss per epoch

### How it works
The loss values stored during training are visualised as two curves over epochs.

### Why this step is necessary
Loss curves provide direct evidence about training dynamics:
- If both losses decrease and stabilise, the model is learning effectively.
- If training loss decreases but validation loss increases, the model is overfitting.
- The epoch at which validation loss is lowest is a strong candidate for the best stopping point.

This plot is explicitly required for selecting an appropriate epoch count.

In [ ]:
# Plot loss curves 

plt.figure(figsize=(7,4))
plt.plot(best["history"]["train_loss"], label="Train loss")
plt.plot(best["history"]["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ANN Training vs Validation Loss")
plt.legend()
plt.show()

## Step 9 — Final evaluation on the test set (generalisation performance)

### What this step does
This step evaluates the selected ANN on the **test set**, reporting:
- macro-F1 score
- classification report (precision/recall/F1 per class)
- confusion matrix

### How it works
- The model produces class logits for each test sample.
- The predicted class is the argmax of logits.
- Metrics compare predictions to true labels.

### Why this step is necessary
The test set is used only once, after model selection, to estimate real-world generalisation. This ensures:
- unbiased performance estimation,
- fair comparison with other models,
- results suitable for reporting and discussion.

Macro-F1 is particularly useful when class frequencies differ, because it weights each class equally rather than being dominated by majority classes.

In [ ]:
# Test set evaluation

device = "cuda" if torch.cuda.is_available() else "cpu"
best_model = best["model"].to(device)
test_loader = make_loader(X_test_s, y_test_i, batch_size=128, shuffle=False)

criterion = nn.CrossEntropyLoss()
test_loss, test_f1 = run_epoch(best_model, test_loader, criterion, optimizer=None, device=device)

# Get detailed predictions 
best_model.eval()
all_true, all_pred = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = best_model(xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_pred.append(preds)
        all_true.append(yb.numpy())

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)

print("\nANN Test loss:", round(test_loss, 4))
print("ANN Test macro-F1:", round(test_f1, 4))
print("\nClassification report:\n", classification_report(all_true, all_pred))

print("\nConfusion matrix:\n", confusion_matrix(all_true, all_pred))